# Application KPI Report Notebook

Notebook wrapper for `Reports/app_mttr_report/generate_app_kpi_report.py`.

Outputs are written under `notebooks/app_mttr_report/Output/`.

## Flow
1. Set env vars manually (Cell 3) or load root `.env` (Cell 4).
2. Configure runtime and output paths (Cell 5).
3. Set report parameters (Cell 6).
4. Run the script (Cell 7).

In [ ]:
import os

os.environ["TEAMSERVER_URL"] = "https://your_saas_instance.contrastsecurity.com/"
os.environ["ORG_UUID"] = ""
os.environ["CONTRAST_AUTH"] = ""
os.environ["CONTRAST_API_KEY"] = ""

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)
print("Loaded .env from ../../.env")

In [ ]:
import os
from pathlib import Path
from datetime import datetime

def get_env(*names):
    for name in names:
        val = os.environ.get(name)
        if val:
            return val
    return None

def find_repo_root(start):
    current = Path(start).resolve()
    for p in [current, *current.parents]:
        if (p / "notebooks").exists() and (p / "README.md").exists():
            return p
    return current

teamserver_url = get_env("TEAMSERVER_URL", "TeamserverURL", "url")
org_uuid = get_env("ORG_UUID", "organizationId")
contrast_auth = get_env("CONTRAST_AUTH", "AUTH", "authHeader")
contrast_api_key = get_env("CONTRAST_API_KEY", "API_KEY", "apiKey")

missing = []
if not teamserver_url: missing.append("TEAMSERVER_URL")
if not org_uuid: missing.append("ORG_UUID")
if not contrast_auth: missing.append("CONTRAST_AUTH")
if not contrast_api_key: missing.append("CONTRAST_API_KEY")
if missing:
    raise EnvironmentError(f"Missing env vars: {missing}")

repo_root = find_repo_root(Path.cwd())
script_path = repo_root / "Reports" / "app_mttr_report" / "generate_app_kpi_report.py"
output_dir = repo_root / "notebooks" / "app_mttr_report" / "Output"
date_str = datetime.now().strftime("%Y-%m-%d")

print("Runtime configuration loaded.")
print(f"Script: {script_path}")
print(f"Output directory: {output_dir}")

In [ ]:
# Set either app_name or app_id
app_name = ""
app_id = ""
days = 365

if not app_name and not app_id:
    raise ValueError("Set app_name or app_id before running the next cell.")

target = app_name if app_name else app_id
safe_target = target.replace(" ", "_").replace("/", "_")
output_path = output_dir / f"{safe_target}_KPI_{date_str}.md"
print(f"Output file: {output_path}")

In [ ]:
import subprocess
import sys

cmd = [sys.executable, str(script_path), "--output", str(output_path), "--days", str(days)]
if app_name:
    cmd += ["--app-name", app_name]
if app_id:
    cmd += ["--app-id", app_id]

result = subprocess.run(cmd, cwd=repo_root, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Script failed with exit code {result.returncode}")
print(f"Done: {output_path}")